# Qdrant: a hands-on implementation

**Track:** Production · **Guide:** [`060-qdrant.md`](../guides/concepts/060-qdrant.md)

Qdrant is a vector database for storing vectors with payload metadata and retrieving nearest neighbors.

> **Intuition:** A map of learned coordinates lets you find nearby ideas, with labels attached to each point.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Qdrant]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Store chunk identifiers and access-control metadata alongside vectors; apply authorization filters at query time.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Define a local in-memory vector collection

Qdrant stores vectors with point IDs and payload metadata. The in-memory client is ephemeral and needs no server.


In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")
collection = "study_passages"
client.create_collection(
    collection_name=collection,
    vectors_config=models.VectorParams(size=3, distance=models.Distance.COSINE),
)

## 2. Upsert vectors with source metadata

Vector and payload are stored together; stable IDs make repeat upserts idempotent.


In [ ]:
client.upsert(
    collection_name=collection,
    points=[
        models.PointStruct(id=1, vector=[0.9, 0.1, 0.0], payload={"text": "car repair guide", "access": "public"}),
        models.PointStruct(id=2, vector=[0.0, 0.1, 0.9], payload={"text": "pasta recipe", "access": "public"}),
        models.PointStruct(id=3, vector=[0.8, 0.2, 0.0], payload={"text": "truck service manual", "access": "staff"}),
    ],
    wait=True,
)

## 3. Search with a payload filter

A query vector must match the collection dimension. Apply metadata authorization filters during retrieval, not after returning data to a caller.


In [ ]:
results = client.query_points(
    collection_name=collection,
    query=[0.95, 0.05, 0.0],
    query_filter=models.Filter(
        must=[models.FieldCondition(key="access", match=models.MatchValue(value="public"))]
    ),
    limit=3,
).points
print([(hit.id, round(hit.score, 3), hit.payload["text"]) for hit in results])
assert all(hit.payload["access"] == "public" for hit in results)
client.close()

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/060-qdrant.md) and the
[learning roadmap](../guides/ROADMAP.md).
